# YUNESA Academic GraphRAG Development

Notebook ini dipakai untuk menguji retrieval setelah KG construction ditulis ke Neo4j AuraDB dan Milvus/Zilliz. Desain mode mengikuti AcademicRAG: `naive`, `subgraph`, `global`, `hybrid`, dan `mix`.

In [ ]:
from pathlib import Path
import importlib.util
import os
import sys

HERE = Path.cwd()
if (HERE / 'src').exists():
    BUILD_GRAPH_DIR = HERE
else:
    BUILD_GRAPH_DIR = Path('notebooks/build-graph').resolve()
sys.path.insert(0, str(BUILD_GRAPH_DIR / 'src'))

required_modules = ['networkx', 'pandas', 'rdflib', 'supabase', 'neo4j', 'pymilvus', 'groq', 'requests']
missing = [module for module in required_modules if importlib.util.find_spec(module) is None]
if missing:
    raise RuntimeError(
        'Notebook dependency belum tersedia di kernel aktif: ' + ', '.join(missing) + '\n'
        'Kernel aktif: ' + sys.executable + '\n'
        'Pilih kernel VS Code: YUNESA Notebooks (.venv), atau jalankan: uv sync --project notebooks'
    )

from yunesa_academic_kg import (
    GraphRAGQueryParam,
    KGConfig,
    format_graphrag_context,
    graphrag_retrieve,
    inspect_milvus_collections,
    inspect_neo4j_graph,
    load_project_env,
)

config = KGConfig.default()
load_project_env(config.project_root)
os.environ.setdefault('NEO4J_TRUST_SELF_SIGNED', '1')
os.environ.setdefault('YUNESA_NEO4J_GRAPH_NAME', 'yunesa_academic_kg')
print(config)
KEYWORD_PROVIDER = os.getenv('YUNESA_GRAPHRAG_KEYWORD_PROVIDER', 'heuristic')
KEYWORD_CACHE = os.getenv('YUNESA_GRAPHRAG_KEYWORD_CACHE', str(BUILD_GRAPH_DIR / 'outputs' / 'academic_kg' / 'graphrag_keyword_cache.json'))
print('Keyword provider:', KEYWORD_PROVIDER)


## 1. Storage Health Check

Pastikan graph store dan vector store sudah terisi sebelum query GraphRAG.

In [ ]:
GRAPH_NAME = os.getenv('YUNESA_NEO4J_GRAPH_NAME', 'yunesa_academic_kg')

neo4j_status = inspect_neo4j_graph(graph_name=GRAPH_NAME)
milvus_status = inspect_milvus_collections()

neo4j_status, milvus_status

## 2. Retrieval Modes

- `naive`: pencarian vector pada `PaperChunk`.
- `subgraph`: pencarian entity vector lalu traversal Neo4j.
- `global`: pencarian relationship embedding.
- `hybrid`: `subgraph + global`.
- `mix`: `naive + keyword + subgraph + global`.

In [ ]:
query = 'paper apa yang membahas retinopati diabetik dengan EfficientNet dan dataset APTOS?'

for mode in ['naive', 'subgraph', 'global', 'hybrid', 'mix']:
    print('\n' + '=' * 100)
    print('MODE:', mode)
    retrieval = graphrag_retrieve(
        query,
        param=GraphRAGQueryParam(mode=mode, top_k=5, graph_name=GRAPH_NAME, keyword_provider=KEYWORD_PROVIDER, keyword_cache_path=KEYWORD_CACHE),
    )
    print(format_graphrag_context(retrieval, max_chars=5000))

## 3. Query Set untuk Evaluasi Manual

Gunakan pertanyaan faktual, relasional, dan sintesis untuk melihat apakah konteks yang diambil sudah memadai sebelum dihubungkan ke LLM.

In [ ]:
test_queries = [
    'Siapa dosen yang menulis publikasi tentang diabetic retinopathy?',
    'Model apa yang digunakan pada paper complaint subcategory prediction?',
    'Dataset apa saja yang digunakan pada publikasi computer vision?',
    'Topik riset apa yang paling sering muncul pada publikasi terbaru?',
]

for q in test_queries:
    retrieval = graphrag_retrieve(q, param=GraphRAGQueryParam(mode='mix', top_k=5, graph_name=GRAPH_NAME, keyword_provider=KEYWORD_PROVIDER, keyword_cache_path=KEYWORD_CACHE))
    print('\nQUERY:', q)
    print(format_graphrag_context(retrieval, max_chars=3000))

In [10]:
query = "Carikan saya paper dari dosen sains data."

retrieval = graphrag_retrieve(
    query,
    param=GraphRAGQueryParam(
        mode="mix",
        top_k=8,
        graph_name=GRAPH_NAME,
        keyword_provider=KEYWORD_PROVIDER,
        keyword_cache_path=KEYWORD_CACHE,
    ),
)

print('Retrieval siap. Jalankan cell berikutnya untuk jawaban chatbot ringkas.')

Retrieval siap. Jalankan cell berikutnya untuk jawaban chatbot ringkas.


In [11]:
from yunesa_academic_kg import GraphRAGGenerationParam, generate_graphrag_answer_with_groq

answer = generate_graphrag_answer_with_groq(
    query=query,
    retrieval=retrieval,
    param=GraphRAGGenerationParam(max_tokens=320, context_max_chars=6500),
)

print(answer['answer'])

print('\nSumber terpakai:')
for title in answer.get('sources', {}).get('paper_titles', [])[:5]:
    print('-', title)


## Direct answer:
Sistem Klasifikasi Limbah Menggunakan Metode Convolutional Neural Network (CNN) Pada Webservice Berbasis Framework Flask oleh Ricky Eka Putra.

## Supporting evidence:
Paper tersebut menggunakan metode Convolutional Neural Network (CNN) dan mencapai akurasi tertinggi 69,77% dengan loss terendah 0,34. Keywords yang terkait dengan paper ini adalah computer science, operating system, humanities, physics, dan Convolutional neural.

Sumber terpakai:
- Sistem Klasifikasi Limbah Menggunakan Metode Convolutional Neural Network (CNN) Pada Webservice Berbasis Framework Flask
- Sistem Informasi Manajemen Sumber Daya Manusia (Studi Kasus Bumida Syariah)
- Aplikasi Ensiklopedia Negara Digital untuk Memotivasi Pengguna dalam Mengenal Negara di Dunia
- Penerapan metode deep learning menggunakan algoritma CNN dengan arsitektur VGG NET untuk pengenalan cuaca
- Severity classification of non-proliferative diabetic retinopathy using convolutional support vector machine
